In [1]:
from pydantic import BaseModel
from typing import Optional # Optional is a type hint that allows for a variable to be either a specific type or None.
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage

from llms_course.utils.llms import get_llm

In [2]:
class State(BaseModel):
    topic: str # This is required by the graph
    joke_setup: Optional[str] = None # So now we do not have to pass it to .invoke() since it is an Optional type
    joke_punchlines: Optional[str] = None # So now we do not have to pass it to .invoke() since it is an Optional type

In [3]:
def generate_joke_setup(state: State) -> State:
    """
    Generate a joke setup for a given topic using an LLM.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()

    system_prompt = """# Role
You are a helpful assistant that generates the setup for jokes based on a topic.

# Instructions
- Output the setup for a joke about the given topic.
- Do not include any punchlines.

# Output Format
- Output the joke setup only without any other text or formatting.
"""

    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = f"Generate a joke setup for a joke about ```{state.topic}```.")
    ]

    response = llm.invoke(messages)
    joke_setup = response.content
    print(joke_setup)
    state.joke_setup = joke_setup
    return state

def generate_joke_punchlines(state: State) -> State:
    """
    Generate three variations of a joke punchline for a given joke setup using an LLM.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    llm = get_llm()
    
    system_prompt = """# Role
You are a helpful assistant that generates the punchlines for jokes based on a setup.

# Instructions
- Output three different punchlines for the given joke setup.

# Output Format
- Output the three punchlines only without any other text or formatting in a numbered list, e.g.
    1. Punchline 1
    2. Punchline 2
    3. Punchline 3
"""

    messages = [
        SystemMessage(content = system_prompt),
        HumanMessage(content = f"Generate three punchlines for the joke setup ```{state.joke_setup}```.")
    ]

    response = llm.invoke(messages)
    punchlines = response.content
    print(punchlines)
    state.joke_punchlines = punchlines
    return state


In [4]:
# Create the graph builder
graph_builder = StateGraph(State)

# Add the nodes to the graph
graph_builder.add_node("generate_joke_setup", generate_joke_setup)
graph_builder.add_node("generate_joke_punchlines", generate_joke_punchlines)

# Add the edges to the graph
graph_builder.add_edge("generate_joke_setup", "generate_joke_punchlines") # From generate_joke_setup -> generate_joke_punchlines
graph_builder.add_edge("generate_joke_punchlines", END) # From generate_joke_punchlines -> END

# Set the entry point to the `generate_joke_setup` node
graph_builder.set_entry_point("generate_joke_setup")

# Compile the graph
graph = graph_builder.compile()

In [5]:
# Input into the graph must match the state schema
inputs = {"topic": "Driving in Lebanon"}

# Invoke the graph with the inputs and return the final state
final_state = graph.invoke(inputs)

print(final_state)

Driving in Lebanon is the only place where a five-minute trip begins with a map, a prayer, and a negotiation with three different drivers.
1. And somehow, you still arrive at a place that’s “just five minutes away”—in Lebanese minutes.

2. By the time you get there, the map has changed, the prayer has been answered, and you’ve made two new enemies.

3. The GPS doesn’t say “recalculating”—it just sighs and asks, “Are you sure you want to do this?”
{'topic': 'Driving in Lebanon', 'joke_setup': 'Driving in Lebanon is the only place where a five-minute trip begins with a map, a prayer, and a negotiation with three different drivers.', 'joke_punchlines': '1. And somehow, you still arrive at a place that’s “just five minutes away”—in Lebanese minutes.\n\n2. By the time you get there, the map has changed, the prayer has been answered, and you’ve made two new enemies.\n\n3. The GPS doesn’t say “recalculating”—it just sighs and asks, “Are you sure you want to do this?”'}
